> [!TIP]
> **Active Development Copy (v2):** This notebook is the workspace for ongoing improvements and extensions to **Business Capability Cost Intelligence & Shared Attribution**.
> The verified baseline remains preserved in `v1_baseline.ipynb` and `../../05_capability_cost_intelligence.ipynb`.

# Notebook 05: Business Capability Cost Intelligence & Shared Attribution

**Series:** Technology Value Intelligence (TBM / ITFM / TVR)  
**Notebook Index:** 05 of 12  
**Focus:** Business Capability Costing, Shared Infrastructure Allocation, Avoiding Double-Counting, Sankey Flow  

---

## 1. Business Problem

CFOs and Business Unit leaders do not purchase "applications" for their own sake; they fund **Business Capabilities** (e.g., Order-to-Cash, Procure-to-Pay, Fraud Detection, Regulatory Reporting). 

When executive leadership asks:
> *"What does our Order-to-Cash capability actually cost?"*

Conventional IT accounting fails because:
1. Capabilities are supported by multiple applications concurrently.
2. A single enterprise application often supports multiple capabilities simultaneously.
3. Common shared infrastructure (networks, storage arrays, security operations) is shared across capabilities without clear boundaries.

Without a rigorous allocation model, organizations suffer from **double-counting** or unallocated financial "slush funds".

---

## 2. Core Concept & Formulation

### 2.1 The Capability Allocation Rule
To achieve 100% financial reconciliation without double-counting:

1. **Direct Application Costs:**
   - If Application $A$ supports a single capability $C$, 100% of $A$'s TCO is assigned to $C$.
   - If Application $A$ supports multiple capabilities, costs are apportioned using normalized weights:
     $$w_i = \begin{cases} 1.0 & \text{if Primary support} \\ 0.5 & \text{if Secondary / Overlapping support} \end{cases}$$
     $$\text{Normalized Weight } \hat{w}_i = \frac{w_i}{\sum_j w_j}$$
     $$\text{Allocated Direct Cost} = \hat{w}_i \times \text{TCO}(A)$$

2. **Shared Infrastructure Costs (Unassigned to specific apps):**
   - Allocated to capabilities proportionally based on their share of total enterprise transaction consumption:
     $$\text{Shared Allocated}(C) = \frac{\text{Transactions}(C)}{\sum_k \text{Transactions}(k)} \times \text{Total Shared Infra Cost}$$

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Dynamically locate project root
workspace_root = Path.cwd()
while workspace_root != workspace_root.parent:
    if (workspace_root / "config.yaml").exists() or (workspace_root / "pyproject.toml").exists():
        break
    workspace_root = workspace_root.parent

src_dir = str(workspace_root / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from tvi.config import load_config
from tvi.cost_analytics import calculate_capability_cost, calculate_application_tco, get_capability_profile
from tvi.database import get_database

config = load_config()
db = get_database(config.database.path)

# Compute complete capability cost attribution
cap_cost_df = calculate_capability_cost()
print("Business Capability Cost Intelligence Compiled:")
display(cap_cost_df[["capability_id", "capability_name", "strategic_priority", "criticality", "supported_apps_count", "direct_app_cost", "shared_infra_allocated", "total_capability_cost", "cost_per_transaction"]].head(8))

Business Capability Cost Intelligence Compiled:


---

## 3. Double-Counting Prevention Audit

We mathematically prove that the sum of all capability allocations reconciles to the exact rupee with total enterprise technology spend in DuckDB:
$$\sum_{c \in \text{Capabilities}} \text{Capability Cost}(c) \equiv \text{Total Enterprise Technology Spend}$$

In [2]:
total_cap_spend = cap_cost_df["total_capability_cost"].sum()
total_direct = cap_cost_df["direct_app_cost"].sum()
total_shared = cap_cost_df["shared_infra_allocated"].sum()
gross_spend = db.get_total_cost()

recon_proof = pd.DataFrame([
    {"Component": "Total Direct App Allocations", "Amount (₹)": total_direct},
    {"Component": "Total Shared Infra Allocations", "Amount (₹)": total_shared},
    {"Component": "Sum of Capability Costs", "Amount (₹)": total_cap_spend},
    {"Component": "Gross Ledger Spend in DuckDB", "Amount (₹)": gross_spend},
    {"Component": "Reconciliation Variance", "Amount (₹)": round(total_cap_spend - gross_spend, 2)}
])
display(recon_proof)
assert abs(total_cap_spend - gross_spend) < 1.0, "Capability spend must perfectly reconcile with total enterprise spend!"
print("✓ Audit PASSED: Zero double-counting. 100% of enterprise technology cost reconciled.")

✓ Audit PASSED: Zero double-counting. 100% of enterprise technology cost reconciled.


---

## 4. Capability Redundancy Cost Quantification (Scenario C)

We isolate capabilities supported by multiple distinct applications to quantify the **cost of functional duplication**:

In [3]:
# Find capabilities with 2+ supporting applications
redundant_caps = cap_cost_df[cap_cost_df["supported_apps_count"] > 1].copy()
redundant_caps["redundant_cost_share"] = (redundant_caps["total_capability_cost"] / total_cap_spend * 100.0).round(1)

print(f"Capabilities with Duplicate Application Coverage ({len(redundant_caps)} capabilities):")
display(redundant_caps[["capability_id", "capability_name", "supported_apps_count", "direct_app_cost", "total_capability_cost", "redundant_cost_share"]])

Capabilities with Duplicate Application Coverage (25 capabilities):


---

## 5. Comparative Allocation Models Analysis

We compare our **Proportional Transaction Allocation Model** against an unweighted **Flat Allocation Model** to observe how allocation mechanics shift capability costs:

In [4]:
# Compare Proportional vs Flat Equal Allocation of Shared Infra
total_shared_cost = float(db.query_df("SELECT SUM(amount) FROM cost_records WHERE application_id IS NULL").iloc[0, 0])
flat_shared_per_cap = total_shared_cost / len(cap_cost_df)

comp_df = cap_cost_df[["capability_id", "capability_name", "direct_app_cost", "shared_infra_allocated"]].copy()
comp_df["flat_shared_allocated"] = round(flat_shared_per_cap, 2)
comp_df["proportional_total"] = comp_df["direct_app_cost"] + comp_df["shared_infra_allocated"]
comp_df["flat_total"] = comp_df["direct_app_cost"] + comp_df["flat_shared_allocated"]
comp_df["allocation_delta_pct"] = (
    (comp_df["proportional_total"] - comp_df["flat_total"]) / comp_df["flat_total"] * 100.0
).round(1)

print("Comparative Allocation Analysis (Top 6 Capabilities):")
display(comp_df[["capability_id", "capability_name", "proportional_total", "flat_total", "allocation_delta_pct"]].head(6))

Comparative Allocation Analysis (Top 6 Capabilities):


---

## 6. Financial Value Flow: Applications to Capabilities to Business Units (Sankey)

We render an interactive Plotly Sankey diagram tracing the flow of technology funding from Applications $\longrightarrow$ Capabilities $\longrightarrow$ Consuming Business Units:

In [5]:
# Compile link data for Sankey
rel_cap = db.query_df("""
    SELECT r.application_id, a.application_name, r.capability_id, c.capability_name, c.business_unit_id, b.business_unit_name
    FROM rel_app_capability r
    JOIN applications a ON r.application_id = a.application_id
    JOIN business_capabilities c ON r.capability_id = c.capability_id
    JOIN business_units b ON c.business_unit_id = b.business_unit_id
""")

# Focus on Top 6 Apps and Top 6 Capabilities to avoid visual clutter
top_app_ids = app_tco_df.head(6)["application_id"].tolist()
rel_top = rel_cap[rel_cap["application_id"].isin(top_app_ids)].copy()

all_nodes = list(rel_top["application_name"].unique()) + list(rel_top["capability_name"].unique()) + list(rel_top["business_unit_name"].unique())
node_map = {name: idx for idx, name in enumerate(all_nodes)}

sources = []
targets = []
values = []

# App -> Cap links
for _, r in rel_top.iterrows():
    sources.append(node_map[r["application_name"]])
    targets.append(node_map[r["capability_name"]])
    values.append(10)

# Cap -> BU links
for _, r in rel_top.drop_duplicates(subset=["capability_name", "business_unit_name"]).iterrows():
    sources.append(node_map[r["capability_name"]])
    targets.append(node_map[r["business_unit_name"]])
    values.append(15)

fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=all_nodes,
        color="#2b5c8f"
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color="rgba(31, 119, 180, 0.4)"
    )
)])

fig.update_layout(title_text="Technology Value Flow: Applications -> Capabilities -> Business Units", font_size=10)
fig.show()

---

## 7. Active Development (v2): Dynamic Capability Costing & Cloud Migration What-If

In the **v2 Architecture**, we introduce configurable allocation dynamics and hierarchical rollups:
1. **Dynamic Configurable Allocation Rules:** Apportion capability costs under user-defined primary/secondary weights and shared drivers.
2. **3-Tier Capability Taxonomy:** Rolls up operational L3 capabilities into L2 functional areas and L1 strategic domains.
3. **Cloud Migration What-If Simulator:** Models financial savings, transitional costs, and payback periods for infrastructure modernization.
4. **Mathematical Reconciliation Audit:** Proves zero double counting across the general ledger and capability catalog.

In [6]:
from tvi.cost_analytics import (
    calculate_dynamic_capability_cost,
    get_nested_capability_hierarchy_cost,
    simulate_cloud_migration_what_if,
    audit_capability_allocation_reconciliation,
)

# 1. Dynamic Allocation Configuration (80/20 Split, Compute Driver)
dyn_cap_cost = calculate_dynamic_capability_cost(primary_weight=0.80, secondary_weight=0.20, shared_driver="compute_hours")
print("Dynamic Capability Cost Allocation (80/20 Split, Compute Driver):")
display(dyn_cap_cost[["capability_id", "capability_name", "total_capability_cost", "direct_app_cost", "shared_infra_allocated"]].head(5))

# 2. 3-Tier Nested Capability Hierarchy
nested_hier = get_nested_capability_hierarchy_cost()
print("\nNested 3-Tier Capability Hierarchy Spend Summary (L1 Domain Rollup):")
l1_rollup = nested_hier.groupby("l1_domain")["total_capability_cost"].sum() / 1e7
display(l1_rollup.reset_index().rename(columns={"total_capability_cost": "Total Spend (₹ Cr)"}))

# 3. Public Cloud Migration What-If Simulator
cloud_what_if = simulate_cloud_migration_what_if(cost_reduction_factor=0.22, transition_cost_pct=0.10)
print("\nWhat-If Financial Simulation: Shared Infrastructure Cloud Modernization:")
for k, v in cloud_what_if.items():
    print(f"  • {k.replace('_', ' ').title()}: {v}")

# 4. Zero Double-Counting Reconciliation Audit
reconcil_audit = audit_capability_allocation_reconciliation()
print(f"\nGeneral Ledger Reconciliation Audit: Status = {reconcil_audit['reconciliation_status']}")
print(f"  Total GL Spend: ₹{reconcil_audit['total_general_ledger_spend']:,}")
print(f"  Total Capability Attributed: ₹{reconcil_audit['total_capability_attributed_spend']:,}")
print(f"  Absolute Delta: ₹{reconcil_audit['absolute_reconciliation_delta']} (Double counting detected: {reconcil_audit['double_counting_detected']})")

Dynamic Capability Cost Allocation (80/20 Split, Compute Driver):

Nested 3-Tier Capability Hierarchy Spend Summary (L1 Domain Rollup):

What-If Financial Simulation: Shared Infrastructure Cloud Modernization:
  • Current Onprem Infra Cr: 9.71
  • Current Cloud Spend Cr: 17.14
  • Total Baseline Infra Cr: 26.85
  • Projected Onprem Cr: 1.46
  • Projected Cloud Cr: 23.58
  • Projected Total Infra Cr: 25.03
  • Annual Gross Savings Cr: 1.82
  • Savings Percentage: 6.76
  • One Time Transition Cost Cr: 0.97
  • Payback Period Months: 6.4

General Ledger Reconciliation Audit: Status = EXACT_RECONCILIATION_PASS
  Total GL Spend: ₹1,618,320,000.12
  Total Capability Attributed: ₹1,618,320,000.11
  Absolute Delta: ₹0.01 (Double counting detected: False)


## 7. Limitations & Analytical Guardrails

1. **Driver Selection:** We utilized digital transaction volume as the proportional driver for shared infrastructure. Other corporate environments might utilize employee headcount, storage volume, or revenue share.
2. **Reconciliation Check:** Summing `total_capability_cost` across all capabilities perfectly reconciles with total enterprise technology spend in DuckDB, ensuring no unallocated variance.

## 8. Next Step

In **Notebook 06: Consumption Intelligence**, we evaluate technology consumption across multiple dimensions, mapping applications into strategic utilization quadrants.